# Criação do Modelo 2 ( corretor )

## PASSO 2) INICIANDO CRIAÇÃO DA BASE DO CORRETOR

### 2.1) Utilização do modelo treinado no experimento do João para a seleção dos Falsos Negativos e Verdadeiros Negativos que serão utilizados no experimento do corretor

In [ ]:
from unidecode import unidecode
import re
import os
import unicodedata
import pandas as pd
from joblib import load
import string
from itertools import product
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt

In [ ]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

In [ ]:
rf_load = load('../../rf_exp2.joblib')

In [ ]:
PATH_BASE_CONCATENADA_BOS_DECCC = os.getenv('PATH_BASE_CONCATENADA_BOS_DECCC')
PATH_BASE_TESTES_FINAIS = os.getenv('PATH_BASE_TESTES_FINAIS')
PATH_GABARITO_CORRETOR = os.getenv('PATH_GABARITO_CORRETOR')


PATH_BASE_FALSOS_NEGATIVOS = os.getenv('PATH_BASE_FALSOS_NEGATIVOS')
PATH_BASE_VERDADEIROS_NEGATIVOS = os.getenv('PATH_BASE_VERDADEIROS_NEGATIVOS')

### 2.1.1) Realizado o carregamento da base com dados resultado da concatenação dos arquivos BOS e DECCC

In [ ]:
base_relato = pd.read_csv(PATH_BASE_CONCATENADA_BOS_DECCC)
base_relato

In [ ]:
base_para_teste = base_relato.sample(frac=0.3, random_state=42)
base_para_teste

In [ ]:
base_para_teste.to_csv(PATH_BASE_TESTES_FINAIS, index=False)

In [ ]:
duplicates = base_relato[~base_relato.isin(base_para_teste)]
duplicates

In [ ]:
base_relato = duplicates.dropna().reset_index(drop=True)
base_relato

In [ ]:
def normalize_text(text):
    if pd.isna(text):
        return text
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(c for c in text if not unicodedata.combining(c))
    return text.upper()

In [ ]:
base_relato['relato'] = base_relato['relato'].apply(normalize_text)
base_relato

### 2.1.3) carregamento do gabarito atualizado

Com a aquisição de novos dados para o experimento o gabarito necessitou de atualização, comportando possiveis tokens novos na base de dados DECCC. Por este motivo, agora há uma diferença entre o gabarito da construção do modelo 1 e o gabarito do modelo dois.


Em tese, não há impacto tão significativo para o experimento do modelo 1. Já que o conteúdo do gabarito sofreu um aumento do volume de dados e não uma redução.

In [ ]:
gab = pd.read_csv(PATH_GABARITO_CORRETOR)
gab.shape

### 2.1.4) Pré-processamento

Realizando o pré-processamento removendo: Stopwords, caracteres especias, números e pontuação dos relatos.

In [ ]:
def preprocess(relato):
    def remove_stop_words(relato):
        # Remove stop words em português
        sw = stopwords.words('portuguese')
        sw = [unidecode(word).upper() for word in sw]
        return ' '.join([token for token in re.split("[, ]", relato) if len(token) > 2 and token not in sw])

    def clean_relato(relato):
        # Remoção de números e pontuação
        relato = re.sub(r'\@(\w*\d?)', " ", relato)
        relato = re.sub(r'([A-z1-9]*_+[A-z1-9]*)|(_+[A-z1-9]*)|([A-z1-9]*_+)', " ", relato)
        relato = re.sub(r"((?:HTTPS?|WWW)(?:\:\/\/?)?[*a-zA-Z0-9\.\/\?\:@\-_=#]+[\.][a-zA-Z]{2,6}[a-zA-Z0-9\.\&\/\?\:@\-_=#]*)", " ", relato)
        relato = re.sub(r'(<[^<>]*>)|(&NBSP;)|(&nbsp;)', " ", relato)
        relato = re.sub(r"[^\w\s]", " ", relato)
        relato = re.sub(r"[\d]", " ", relato)
        relato = remove_stop_words(relato)
        return relato
    
    relato = clean_relato(relato)
    return relato

verificação do sucesso do pré-processamento

In [ ]:
base_relato['relato_tratado'] = base_relato['relato'].apply(preprocess)
base_relato

### EDA - Tanto da Base para o treinamento quanto para a base dos testes

In [ ]:
teste_base_relato = base_relato.copy()

def create_df(text):
    words = text.split()
    return pd.DataFrame({'pred_palavra': words})

teste1 = pd.concat(teste_base_relato['relato_tratado'].apply(create_df).tolist(), ignore_index=True)


In [ ]:
teste2 = teste1.merge(gab[['palavra', 'y']], left_on='pred_palavra', right_on='palavra', how='left')


#### totas as ocorrências dos tokens

In [ ]:
teste2.value_counts('y')

In [ ]:
teste2[teste2['y'] == 0].shape

In [ ]:
teste2[teste2['y'] == 1].shape

#### desencargo de conciência dos resultados para os testess

In [ ]:
base_para_teste['relato_tratado'] = base_para_teste['relato'].apply(preprocess)

In [ ]:
teste_base_relato = base_para_teste.copy()

def create_df(text):
    words = text.split()
    return pd.DataFrame({'pred_palavra': words})

teste1 = pd.concat(teste_base_relato['relato_tratado'].apply(create_df).tolist(), ignore_index=True)


In [ ]:
teste2 = teste1.merge(gab[['palavra', 'y']], left_on='pred_palavra', right_on='palavra', how='left')

numero total de TODAS AS OCORRÊNCIAS DOS REGISTROS

In [ ]:
teste2[teste2['y'] == 0].shape

In [ ]:
teste2[teste2['y'] == 1].shape

olha como fica com o mesmo resultado da matriz

In [ ]:
teste2[teste2['y'] == 0].drop_duplicates().shape

In [ ]:
teste2[teste2['y'] == 1].drop_duplicates().shape

### 2.1.5) Identificação dos falsos negativos

Neste trecho é realizada a identificação e criação de um DataFrame com os falsos negativos (tokens classificados como palavra '0', pelo modelo, que na verdade são nomes '1', no gabarito) após a predição dos tokens do relato relato

In [ ]:
def false_negative(df_bop: pd.DataFrame):
    fn_list = []

    for n_bop, text in zip(df_bop['n_bop'], df_bop['relato_tratado']):
        text = text.split()
        df = pd.DataFrame(text, columns=['palavra'])
        
        df['y_palavra'] = rf_load.predict(df['palavra'].to_list())

        df['predecessor'] = df['palavra'].shift(1).fillna('#')
        df['y_predecessor'] = df['predecessor'].apply(lambda x: df[df['palavra'] == x]['y_palavra'].iloc[0] if not df[df['palavra'] == x].empty else 2)
        
        df['sucessor'] = df['palavra'].shift(-1).fillna('#')
        df['y_sucessor'] = df['sucessor'].apply(lambda x: df[df['palavra'] == x]['y_palavra'].iloc[0] if not df[df['palavra'] == x].empty else 2)
        
        for _, row in df.iterrows():
            y_true_row = gab[gab['palavra'] == row['palavra']]
            y_true = y_true_row['y'].iloc[0] if not y_true_row.empty else 0
            
            if row['y_palavra'] == 0 and y_true == 1:
                fn_list.append({
                    'predecessor': row['predecessor'],
                    'y_predecessor': int(row['y_predecessor']),
                    'palavra': row['palavra'],
                    'y_palavra': int(row['y_palavra']),
                    'sucessor': row['sucessor'],
                    'y_sucessor': int(row['y_sucessor']),
                    'n_bop': n_bop
                })
                
    return pd.DataFrame(fn_list)


### 2.1.6) Identificação dos verdadeiros negativos

Neste trecho é realizada a identificação e criação de um DataFrame com os verdadeiros negativos (tokens classificados como palavra '0', pelo modelo, e que realmente são palavras '0', no gabarito) após a predição dos tokens do relato

In [ ]:
def true_negative(df_bop: pd.DataFrame):
    tn_list = []

    for n_bop, text in zip(df_bop['n_bop'], df_bop['relato_tratado']):
        text = text.split()
        df = pd.DataFrame(text, columns=['palavra'])
        
        df['y_palavra'] = rf_load.predict(df['palavra'].to_list())

        df['predecessor'] = df['palavra'].shift(1).fillna('#')
        df['y_predecessor'] = df['predecessor'].apply(lambda x: df[df['palavra'] == x]['y_palavra'].iloc[0] if not df[df['palavra'] == x].empty else 2)
        
        df['sucessor'] = df['palavra'].shift(-1).fillna('#')
        df['y_sucessor'] = df['sucessor'].apply(lambda x: df[df['palavra'] == x]['y_palavra'].iloc[0] if not df[df['palavra'] == x].empty else 2)
        
        for _, row in df.iterrows():
            y_true_row = gab[gab['palavra'] == row['palavra']]
            y_true = y_true_row['y'].iloc[0] if not y_true_row.empty else 0
            
            if row['y_palavra'] == 0 and y_true == 0:
                tn_list.append({
                    'predecessor': row['predecessor'],
                    'y_predecessor': int(row['y_predecessor']),
                    'palavra': row['palavra'],
                    'y_palavra': int(row['y_palavra']),
                    'sucessor': row['sucessor'],
                    'y_sucessor': int(row['y_sucessor']),
                    'n_bop': n_bop
                })
    return pd.DataFrame(tn_list)


In [ ]:
tn_base_relatos = true_negative(base_relato)
fn_base_relatos = false_negative(base_relato)

### 2.1.7) Tratamentos das celulas vazias

Surgiu um desafio na etapa da realização da captura dos predecessores e sucessores. Foram identificados, em alguns casos, a inexistencia de um predecessor ou sucessor do token classificado. Por exemplo, 'RELATOR', que se localiza no início dos textos, é um VERDADEIRO NEGATIVO, e por ser a primeira palavra do vetor de tokens não possui predecessor. Também ocorreram problemas na captura dos FALSOS NEGATIVOS, ouve a ocorrência de dois nomes que se localizavam no fim do vetor de tokens, estas não possuiam sucessores.

Qual o problema? Devido a falta do valor do preditor acerca da palavra inexistente, ele armazena apenas '' (vazio). Isso é um problema já que os modelos utilizados trabalham apenas com dados numéricos.

Estas ocorrências, por mais raras que possa parecer, são desafios que nosso modelo deve estar preparado, visto que há sim grandes chances de ele se deparar com situações como esta em um ambiente real.

Diante disso, foi elaborada uma estratégia para tratar este erro.

Foi implementada uma nova categoria, justamente para solucionar este problema. Assim, palavras que não possuem predecessor ou sucessores, receberão como valor para os respectivos vizinhos o número inteiro '2'.

Logo, temos estas categorias para os tokens de predecessor ou sucessores:

0 - palavra  
1 - nome  
2 - predecessor ou sucessor inexistente  

##### A presença do caracter coringa

A adição do caracter coringa ocorre quando um determinado token não possue um sucessor ou predecessor. Esse evento ocorre quando se dá o início do texto e o fim do mesmo. Podemos identificar pelas ocorrências de palavras que estão sendo inseridas no início dos relatos. Não necessáriamente será realmente a primeira ou a última palavra do texto preprocessado o motivo é simples. O pré-processamento irá remover stopwords, logo não podemos nos basear pelo relato antes do pré-processamento, mas sim pelo relato após as motificações.

In [ ]:
fn_base_relatos[fn_base_relatos['y_predecessor'] == 2]

In [ ]:
tn_base_relatos[tn_base_relatos['y_predecessor'] == 2]

In [ ]:
fn_base_relatos[fn_base_relatos['y_sucessor'] == 2]

In [ ]:
tn_base_relatos[tn_base_relatos['y_sucessor'] == 2]

### 2.1.8) Seleção dos VERDADEIROS NEGATIVOS para o balanceamento da base

Aqui é realizada a seleção pseudo-aleatória das amostras da base de VERDADEIROS NEGATIVOS, isso é realizado pelo fato da base de verdadeiros negativos ser muito superior a base de FALSOS NEGATIVOS.  

Logo, para que ambas as bases possuam o mesmo número de amostras, foi realizada a extração 'aleatória' de 406  amostras da base de VERDADEIROS NEGATIVOS. (Faço isso aqui, pra melhorar o desempenho kakakakka)

## PASSO 3) Criação do Vetor de Bigramas dos Falsos Negativos e Verdadeiros Negativos

Este é o mesmo modelo de vetor de bigramas dos experimentos anteriores, podes visualizar logo mais a frente.

In [ ]:
def bigrams_vector(lista_fn):
    alfabeto = list(string.ascii_uppercase)
    alfabeto_bigrams = [''.join(bigram) for bigram in list(product(alfabeto, repeat=2))]
    vectorizer = CountVectorizer(analyzer='char', vocabulary=alfabeto_bigrams, lowercase=False, ngram_range=(2, 2))
    vector = vectorizer.fit_transform(lista_fn)
    return pd.DataFrame(vector.todense(), columns=alfabeto_bigrams), vectorizer

In [ ]:
palavra = fn_base_relatos['palavra']
fn_vector_bigrams, vectorizer = bigrams_vector(palavra.to_list())
fn_vector_bigrams

In [ ]:
palavra = tn_base_relatos['palavra']
tn_vector_bigrams, vectorizer = bigrams_vector(palavra.to_list())
tn_vector_bigrams

## PASSO 4) Concatenação DA PREDIÇÃO DOS FNs e TNs coletados

adiciono também os tokens pra vefificação do que foi predito

In [ ]:
fn_vector_bigrams = pd.concat([fn_vector_bigrams, fn_base_relatos[['predecessor', 'y_predecessor', 'palavra','sucessor','y_sucessor']]], axis=1)
fn_vector_bigrams

In [ ]:
tn_vector_bigrams = pd.concat([tn_vector_bigrams, tn_base_relatos[['predecessor', 'y_predecessor', 'palavra','sucessor','y_sucessor']]], axis=1)
tn_vector_bigrams

## Passo 5) Realização da Classificação Gramatical dos Falsos Negativos e Verdadeiros Positivos

In [ ]:
import spacy as scy
from sklearn.preprocessing import LabelEncoder

In [ ]:
nlp = scy.load('pt_core_news_lg')

In [ ]:
lb = LabelEncoder()

### 5.1) Criação dos labels da classificação gramatical

No código que aproveitei do classificador do spacy, possuia a renomeação dos labels para 'pt-br', digamos assim. Por essa questão, não alterei já que a visualização se torna bem melhor para validação dos resultados exibidos pelo classificador, então foram criadas 14 labels para a classificação, estes estão listado abaixo:

In [ ]:
labels = ['<SUBSTANTIVO>', '<VERBO>', '<ADJETIVO>', '<ADVERBIO>', '<PONTUACAO>',
            '<CONJUNCAO>', '<PRONOME>', '<ARTIGO>', '<PREPOSIÇÃO>', '<NUMERAL>',
            '<SIMBOLO>', '<PALAVRA ESTRANGEIRA>', '<ESPACO>', '<NULL>']

In [ ]:
lb.fit(labels)

In [ ]:
lb.classes_

adicionar a modificação pra deixar esse código melhor o map que foi feito na etapa de testes do experimento do corretor

In [ ]:
def renomear(doc):

    csf = []
    for token in doc:
        if token.pos_ == 'NOUN' or token.pos_ == 'PROPN':
            csf.append('<SUBSTANTIVO>')
        elif token.pos_ == 'VERB' or token.pos_ == 'AUX':
            csf.append('<VERBO>')
        elif token.pos_ == 'ADJ':
            csf.append('<ADJETIVO>')
        elif token.pos_ == 'ADV':
            csf.append('<ADVERBIO>')
        elif token.pos_ == 'PUNCT':
            csf.append('<PONTUACAO>')
        elif token.pos_ == 'CCONJ' or token.pos_ == 'SCONJ':
            csf.append('<CONJUNCAO>')
        elif token.pos_ == 'PRON':
            csf.append('<PRONOME>')
        elif token.pos_ == 'DET':
            csf.append('<ARTIGO>')
        elif token.pos_ == 'ADP':
            csf.append('<PREPOSIÇÃO>')
        elif token.pos_ == 'NUM':
            csf.append('<NUMERAL>')
        elif token.pos_ == 'SYM':
            csf.append('<SIMBOLO>')
        elif token.pos_ == 'X':
            csf.append('<PALAVRA ESTRANGEIRA>')
        elif token.pos_ == 'SPACE':
            csf.append('<ESPACO>')
    return csf


### 5.2) Tratamento dos valores vazios de predecessores e sucessores

Sim, teve dor de cabeça para os predecessores e sucessores inexistentes aqui também. Este foi solucionado de maneira mais simples.

Se observar na lista anterior de labels, podes identificar um label que não é encontrado na função de renomear, aqui em cima. Esse carinha é o <NULL>

Esse cara será a nossa classificação gramatical para os nulos, já que não tem como classificar os mesmos com nada além disso, já que eles não existem.

Logo, aqui a baixo podes ler:

Se token == '#':
    adicione na lista word_class <NULL>

tratando os nossos predecessores e sucessores vazios

In [ ]:
def create_base_cg(lista_tokens):
    word_class = []
    for token in lista_tokens:
        if token == '#':
            word_class.append('<NULL>')
        else:
            lreview = token.lower()
            ldoc = nlp(lreview)
            classes = renomear(ldoc)
            for i in range(len(token.split())):
                word_class.append(classes[i])

    return pd.DataFrame({'palavra':lista_tokens, 'classe':word_class})

Aqui é onde realizo a chamada da função que irá realizar a classificação de todo mundo, tanto da palavra, quanto do predecessor e sucessor. Adicionando a mesma juntamente com o vetor de bigramas e a predição do predecessor e sucessor.

In [ ]:
fn_vector_bigrams['cg_p'] = create_base_cg(fn_base_relatos['predecessor'])['classe']
fn_vector_bigrams['cg_w'] = create_base_cg(fn_base_relatos['palavra'])['classe']
fn_vector_bigrams['cg_s'] = create_base_cg(fn_base_relatos['sucessor'])['classe']

In [ ]:
fn_vector_bigrams['cg_p'] = lb.transform(fn_vector_bigrams['cg_p'])
fn_vector_bigrams['cg_w'] = lb.transform(fn_vector_bigrams['cg_w'])
fn_vector_bigrams['cg_s'] = lb.transform(fn_vector_bigrams['cg_s'])
fn_vector_bigrams

In [ ]:
tn_vector_bigrams['cg_p'] = create_base_cg(tn_base_relatos['predecessor'])['classe']
tn_vector_bigrams['cg_w'] = create_base_cg(tn_base_relatos['palavra'])['classe']
tn_vector_bigrams['cg_s'] = create_base_cg(tn_base_relatos['sucessor'])['classe']

In [ ]:
tn_vector_bigrams['cg_p'] = lb.transform(tn_vector_bigrams['cg_p'])
tn_vector_bigrams['cg_w'] = lb.transform(tn_vector_bigrams['cg_w'])
tn_vector_bigrams['cg_s'] = lb.transform(tn_vector_bigrams['cg_s'])
tn_vector_bigrams

In [ ]:
for index, classe in zip(range(0, len(lb.classes_)), lb.classes_):
    print(str(index) + ' ' + classe)

### 5.3) Drop duplicates

remoção dos dados duplicados

In [ ]:
fn_vector_bigrams.shape

In [ ]:
fn_vector_bigrams = fn_vector_bigrams.drop_duplicates()
fn_vector_bigrams.shape

In [ ]:
fn_base_final = fn_vector_bigrams
# fn_base_final = fn_vector_bigrams.drop(['sucessor', 'predecessor','palavra'], axis=1)
# fn_base_final

In [ ]:
fn_base_final

#### Coleta de token

In [ ]:
test_alfabeto = list(string.ascii_uppercase)
test_alfabeto_bigrams = [''.join(bigram) for bigram in list(product(test_alfabeto, repeat=2))]

teste = fn_base_final.iloc[[268]]

test_df_result = teste.merge(fn_base_final, on=test_alfabeto_bigrams, how='left')
test_df_result

In [ ]:
fn_base_final = fn_vector_bigrams.drop(['sucessor', 'predecessor','palavra'], axis=1)
fn_base_final

In [ ]:
tn_vector_bigrams = tn_vector_bigrams.drop_duplicates()
tn_vector_bigrams.shape

In [ ]:
tn_base_final = tn_vector_bigrams.drop(['sucessor', 'predecessor','palavra'], axis=1)
tn_base_final

### 5.4) Armazenamento das bases prontas para treino dos Falsos Negativos e Verdadeiros Negativos

In [ ]:
fn_base_final.to_csv(PATH_BASE_FALSOS_NEGATIVOS, index=False)
tn_base_final.to_csv(PATH_BASE_VERDADEIROS_NEGATIVOS, index=False)

no fim temos estes dois arquivos

duas tabelas contendo

Parte 1: Vetor de bigramas do token (O FALSO NEGATIVO OU VERDADEIRO NEGATIVO)  
Parte 2: A resultado da predição do nosso modelo para com os predecessores e sucessores do token representado no vetor de bigramas  
Parte 3: A classificação gramatical do predecessor, da token em si e do sucessor deste token.  

In [ ]:
base_fn = pd.read_csv(PATH_BASE_FALSOS_NEGATIVOS)
base_fn

In [ ]:
base_tn = pd.read_csv(PATH_BASE_VERDADEIROS_NEGATIVOS)
base_tn

## PASSO 6) Realização dos testes na base nova

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score,  recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

Realizando o carregamento das bases de falsos negativos e verdadeiros negativos criada no arquivo 'criacao_base_fn.ipynb'

In [ ]:
fn_base = pd.read_csv(PATH_BASE_FALSOS_NEGATIVOS)
tn_base = pd.read_csv(PATH_BASE_VERDADEIROS_NEGATIVOS)

setando para os valores de y para o modelo

fn_base que são nossos NOMES que foram classificados como PALAVRAS recebem o rótulo '1'  
tn_base que são nossas PALAVRAS que foram classificados corretamente como PALAVRAS recebem o rótulo '0'

In [ ]:
fn_base['y_true'] = 1
tn_base['y_true'] = 0

In [ ]:
len(tn_base) / len(fn_base)

In [ ]:
df2 = tn_base.copy()

In [ ]:
count = 0
scores_kfolds = {'Accuracy' : [], 'Recall': [], 'F1-score':[], 'Precision':[]}

### Tests K-FOLDs ( Random Forest ) 

In [ ]:
while(len(df2) >= len(fn_base)):
    buffer_df_tn = df2.sample(n=len(fn_base), random_state=42)
    
    # contat
    base_final = pd.concat([fn_base, buffer_df_tn])

    # resetando o index da base final
    base_final = base_final.reset_index(drop=True)
    
    # labels
    y = base_final['y_true']
    X = base_final.drop(['y_true'], axis=1)

    ## 6.1) Realização da divisão da base para treino e teste - (30% para treino 70% para teste)
    """
    aqui é realizada a divisão da base para treino e teste

    estabelecendo o critério de divisão 30-70

    30% da base sendo utilizada para teste  
    70% da base sendo utilizada para treinamento 

    """

    # separação treino e teste
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

    def tabela_divisao(y_train, y_test):
        treino = pd.DataFrame(y_train.value_counts()).rename({'count': 'Treinamento'}, axis=1).rename_axis(None, axis=0)
        teste = pd.DataFrame(y_test.value_counts()).rename({'count': 'Teste'}, axis=1).rename_axis(None, axis=0)
        df_dist = pd.concat([treino, teste], axis=1)
        df_dist['Total'] = df_dist.sum(axis=1)
        pd.concat([df_dist, df_dist.sum().to_frame('Total').T]).T
        return df_dist.T
    
    tabela_divisao(y_train, y_test)

    y_predictions = {}

    # selecionando Random Forest para treino e teste
    rf = RandomForestClassifier()
    rf.fit(X_train, y_train)

    y_predictions['Random Forest'] = rf.predict(X_test)
    scores_kfolds['Accuracy'].append(accuracy_score(y_test, y_predictions['Random Forest']))
    scores_kfolds['Recall'].append(recall_score(y_test, y_predictions['Random Forest']))
    scores_kfolds['F1-score'].append(f1_score(y_test, y_predictions['Random Forest']))
    scores_kfolds['Precision'].append(precision_score(y_test, y_predictions['Random Forest']))
    
    # dropando para reiniciar o ciclo
    df2 = df2.drop(buffer_df_tn.index)
    count+=1
    print("Count: " + str(count) + " len df: " + str(df2.shape))

In [ ]:
import numpy as  np
# média dos resultados
metrica = 0
for metrica in scores_kfolds.keys(): 
    print(metrica + ": " + str(np.mean(scores_kfolds[metrica])))

In [ ]:
# desvio padrao dos resultados
idx_metrica = 0
for idx_metrica in scores_kfolds.keys(): 
    print(idx_metrica + ": " + str(np.std(scores_kfolds[idx_metrica])))

Detalhe

In [ ]:
plt.figure(figsize=(10, 4))
plt.boxplot(scores_kfolds.values(), labels=scores_kfolds.keys(), patch_artist=True)
plt.grid(True)
plt.title('ReName Model - Random Forest (K-Folds TNs)')
plt.ylabel('Results (%)')
# plt.savefig('images/boxplot.png', bbox_inches='tight', dpi = 300)
plt.show()

Geral

In [ ]:
plt.figure(figsize=(10, 4))
plt.boxplot(scores_kfolds.values(), labels=scores_kfolds.keys(), patch_artist=True)
plt.grid(True)
plt.ylim(0.0, 1.0)
plt.title('Random Forest - (K-Folds TNs)')
plt.ylabel('Results (%)')
# plt.savefig('images/boxplot.png', bbox_inches='tight', dpi = 300)
plt.show()

### Tests K-FOLDs ( Support Vector Machine )

In [ ]:
df2 = tn_base.copy()
count = 0
scores_kfolds = {'acuracia' : [], 'recall': [], 'f1_score':[], 'precisao':[]}

In [ ]:
while(len(df2) >= len(fn_base)):
    buffer_df_tn = df2.sample(n=len(fn_base), random_state=42)
    
    # contat
    base_final = pd.concat([fn_base, buffer_df_tn])

    # resetando o index da base final
    base_final = base_final.reset_index(drop=True)
    
    # labels
    y = base_final['y_true']
    X = base_final.drop(['y_true'], axis=1)

    ## 6.1) Realização da divisão da base para treino e teste - (30% para treino 70% para teste)
    """
    aqui é realizada a divisão da base para treino e teste

    estabelecendo o critério de divisão 30-70

    30% da base sendo utilizada para teste  
    70% da base sendo utilizada para treinamento 

    """

    # separação treino e teste
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

    def tabela_divisao(y_train, y_test):
        treino = pd.DataFrame(y_train.value_counts()).rename({'count': 'Treinamento'}, axis=1).rename_axis(None, axis=0)
        teste = pd.DataFrame(y_test.value_counts()).rename({'count': 'Teste'}, axis=1).rename_axis(None, axis=0)
        df_dist = pd.concat([treino, teste], axis=1)
        df_dist['Total'] = df_dist.sum(axis=1)
        pd.concat([df_dist, df_dist.sum().to_frame('Total').T]).T
        return df_dist.T
    
    tabela_divisao(y_train, y_test)

    y_predictions = {}

    # selecionando Support Vector Machine para treino e teste
    from sklearn.svm import SVC

    clf = SVC(gamma='auto', random_state=42)
    clf.fit(X_train, y_train)

    y_predictions['SVM'] = clf.predict(X_test)
    scores_kfolds['acuracia'].append(accuracy_score(y_test, y_predictions['SVM']))
    scores_kfolds['recall'].append(recall_score(y_test, y_predictions['SVM']))
    scores_kfolds['f1_score'].append(f1_score(y_test, y_predictions['SVM']))
    scores_kfolds['precisao'].append(precision_score(y_test, y_predictions['SVM']))
    
    # dropando para reiniciar o ciclo
    df2 = df2.drop(buffer_df_tn.index)
    count+=1
    print("Count: " + str(count) + " len df: " + str(df2.shape))

In [ ]:
import numpy as  np
# média dos resultados
metrica = 0
for metrica in scores_kfolds.keys(): 
    print(metrica + ": " + str(np.mean(scores_kfolds[metrica])))

In [ ]:
# desvio padrao dos resultados
idx_metrica = 0
for idx_metrica in scores_kfolds.keys(): 
    print(idx_metrica + ": " + str(np.std(scores_kfolds[idx_metrica])))

Detalhes

In [ ]:
plt.figure(figsize=(10, 4))
plt.boxplot(scores_kfolds.values(), labels=scores_kfolds.keys(), patch_artist=True)
plt.grid(True)
plt.title('Support Vector Machine - (K-Folds TNs)')
plt.ylabel('Results (%)')
# plt.savefig('images/boxplot.png', bbox_inches='tight', dpi = 300)
plt.show()

Geral

In [ ]:
plt.figure(figsize=(10, 4))
plt.boxplot(scores_kfolds.values(), labels=scores_kfolds.keys(), patch_artist=True)
plt.grid(True)
plt.ylim(0.0, 1.0)
plt.title('Support Vector Machine - (K-Folds TNs)')
plt.ylabel('Results (%)')
# plt.savefig('images/boxplot.png', bbox_inches='tight', dpi = 300)
plt.show()

### Logistic Regression ( K-Folds TNs )

In [ ]:
df2 = tn_base.copy()
count = 0
scores_kfolds = {'acuracia' : [], 'recall': [], 'f1_score':[], 'precisao':[]}

In [ ]:
while(len(df2) >= len(fn_base)):
    buffer_df_tn = df2.sample(n=len(fn_base), random_state=42)
    
    # contat
    base_final = pd.concat([fn_base, buffer_df_tn])

    # resetando o index da base final
    base_final = base_final.reset_index(drop=True)
    
    # labels
    y = base_final['y_true']
    X = base_final.drop(['y_true'], axis=1)

    ## 6.1) Realização da divisão da base para treino e teste - (30% para treino 70% para teste)
    """
    aqui é realizada a divisão da base para treino e teste

    estabelecendo o critério de divisão 30-70

    30% da base sendo utilizada para teste  
    70% da base sendo utilizada para treinamento 

    """

    # separação treino e teste
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

    def tabela_divisao(y_train, y_test):
        treino = pd.DataFrame(y_train.value_counts()).rename({'count': 'Treinamento'}, axis=1).rename_axis(None, axis=0)
        teste = pd.DataFrame(y_test.value_counts()).rename({'count': 'Teste'}, axis=1).rename_axis(None, axis=0)
        df_dist = pd.concat([treino, teste], axis=1)
        df_dist['Total'] = df_dist.sum(axis=1)
        pd.concat([df_dist, df_dist.sum().to_frame('Total').T]).T
        return df_dist.T
    
    tabela_divisao(y_train, y_test)

    y_predictions = {}

    # selecionando Logistc Regression para treino e teste
    from sklearn.linear_model import LogisticRegression

    model = LogisticRegression(random_state=42)
    model.fit(X_train, y_train)

    y_predictions['Logistc Regression'] = model.predict(X_test)
    scores_kfolds['acuracia'].append(accuracy_score(y_test, y_predictions['Logistc Regression']))
    scores_kfolds['recall'].append(recall_score(y_test, y_predictions['Logistc Regression']))
    scores_kfolds['f1_score'].append(f1_score(y_test, y_predictions['Logistc Regression']))
    scores_kfolds['precisao'].append(precision_score(y_test, y_predictions['Logistc Regression']))
    
    # dropando para reiniciar o ciclo
    df2 = df2.drop(buffer_df_tn.index)
    count+=1
    print("Count: " + str(count) + " len df: " + str(df2.shape))

In [ ]:
import numpy as  np
# média dos resultados
metrica = 0
for metrica in scores_kfolds.keys(): 
    print(metrica + ": " + str(np.mean(scores_kfolds[metrica])))

In [ ]:
# desvio padrao dos resultados
idx_metrica = 0
for idx_metrica in scores_kfolds.keys(): 
    print(idx_metrica + ": " + str(np.std(scores_kfolds[idx_metrica])))

Detalhes

In [ ]:
plt.figure(figsize=(10, 4))
plt.boxplot(scores_kfolds.values(), labels=scores_kfolds.keys(), patch_artist=True)
plt.grid(True)
plt.title('Logistc Regression - (K-Folds TNs)')
plt.ylabel('Results (%)')
# plt.savefig('images/boxplot.png', bbox_inches='tight', dpi = 300)
plt.show()

Geral

In [ ]:
plt.figure(figsize=(10, 4))
plt.boxplot(scores_kfolds.values(), labels=scores_kfolds.keys(), patch_artist=True)
plt.grid(True)
plt.ylim(0.0, 1.0)
plt.title('Logistc Regression - (K-Folds TNs)')
plt.ylabel('Results (%)')
# plt.savefig('images/boxplot.png', bbox_inches='tight', dpi = 300)
plt.show()

### Gradient Boosting ( K-Folds TNs )

In [ ]:
df2 = tn_base.copy()
count = 0
scores_kfolds = {'acuracia' : [], 'recall': [], 'f1_score':[], 'precisao':[]}

In [ ]:
while(len(df2) >= len(fn_base)):
    buffer_df_tn = df2.sample(n=len(fn_base), random_state=42)
    
    # contat
    base_final = pd.concat([fn_base, buffer_df_tn])

    # resetando o index da base final
    base_final = base_final.reset_index(drop=True)
    
    # labels
    y = base_final['y_true']
    X = base_final.drop(['y_true'], axis=1)

    ## 6.1) Realização da divisão da base para treino e teste - (30% para treino 70% para teste)
    """
    aqui é realizada a divisão da base para treino e teste

    estabelecendo o critério de divisão 30-70

    30% da base sendo utilizada para teste  
    70% da base sendo utilizada para treinamento 

    """

    # separação treino e teste
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

    def tabela_divisao(y_train, y_test):
        treino = pd.DataFrame(y_train.value_counts()).rename({'count': 'Treinamento'}, axis=1).rename_axis(None, axis=0)
        teste = pd.DataFrame(y_test.value_counts()).rename({'count': 'Teste'}, axis=1).rename_axis(None, axis=0)
        df_dist = pd.concat([treino, teste], axis=1)
        df_dist['Total'] = df_dist.sum(axis=1)
        pd.concat([df_dist, df_dist.sum().to_frame('Total').T]).T
        return df_dist.T
    
    tabela_divisao(y_train, y_test)

    y_predictions = {}

    # selecionando Gradient Boosting para treino e teste
    from sklearn.ensemble import GradientBoostingClassifier
    gb = GradientBoostingClassifier(random_state=42)
    gb.fit(X_train, y_train)

    y_predictions['Gradient Boosting'] = gb.predict(X_test)
    scores_kfolds['acuracia'].append(accuracy_score(y_test, y_predictions['Gradient Boosting']))
    scores_kfolds['recall'].append(recall_score(y_test, y_predictions['Gradient Boosting']))
    scores_kfolds['f1_score'].append(f1_score(y_test, y_predictions['Gradient Boosting']))
    scores_kfolds['precisao'].append(precision_score(y_test, y_predictions['Gradient Boosting']))
    
    # dropando para reiniciar o ciclo
    df2 = df2.drop(buffer_df_tn.index)
    count+=1
    print("Count: " + str(count) + " len df: " + str(df2.shape))

In [ ]:
import numpy as  np
# média dos resultados
metrica = 0
for metrica in scores_kfolds.keys(): 
    print(metrica + ": " + str(np.mean(scores_kfolds[metrica])))

In [ ]:
# desvio padrao dos resultados
idx_metrica = 0
for idx_metrica in scores_kfolds.keys(): 
    print(idx_metrica + ": " + str(np.std(scores_kfolds[idx_metrica])))

Detalhes

In [ ]:
plt.figure(figsize=(10, 4))
plt.boxplot(scores_kfolds.values(), labels=scores_kfolds.keys(), patch_artist=True)
plt.grid(True)
plt.title('Gradient Boosting - (K-Folds TNs)')
plt.ylabel('Results (%)')
# plt.savefig('images/boxplot.png', bbox_inches='tight', dpi = 300)
plt.show()

Geral

In [ ]:
plt.figure(figsize=(10, 4))
plt.boxplot(scores_kfolds.values(), labels=scores_kfolds.keys(), patch_artist=True)
plt.grid(True)
plt.ylim(0.0, 1.0)
plt.title('Gradient Boosting - (K-Folds TNs)')
plt.ylabel('Results (%)')
# plt.savefig('images/boxplot.png', bbox_inches='tight', dpi = 300)
plt.show()

### Criaçao do modelo ( RANDOM FOREST )

In [ ]:
tn_create_model = tn_base.sample(n=len(fn_base), random_state=42)
tn_create_model = tn_create_model.drop_duplicates(ignore_index=True)

# contat
base_final = pd.concat([fn_base, tn_create_model])

# resetando o index da base final
base_final = base_final.reset_index(drop=True)

# labels
y = base_final['y_true']
X = base_final.drop(['y_true'], axis=1)

## 6.1) Realização da divisão da base para treino e teste - (30% para treino 70% para teste)
"""
aqui é realizada a divisão da base para treino e teste

estabelecendo o critério de divisão 30-70

30% da base sendo utilizada para teste  
70% da base sendo utilizada para treinamento 

"""

# separação treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state = 42)

def tabela_divisao(y_train, y_test):
    treino = pd.DataFrame(y_train.value_counts()).rename({'count': 'Treinamento'}, axis=1).rename_axis(None, axis=0)
    teste = pd.DataFrame(y_test.value_counts()).rename({'count': 'Teste'}, axis=1).rename_axis(None, axis=0)
    df_dist = pd.concat([treino, teste], axis=1)
    df_dist['Total'] = df_dist.sum(axis=1)
    pd.concat([df_dist, df_dist.sum().to_frame('Total').T]).T
    return df_dist.T

tabela_divisao(y_train, y_test)

y_predictions = {}

#### 6.2) Realização do treino e teste dos modelos utilizando os modelos presentes no artigo.

aqui é realizado o carregamento dos algoritmos de classificação e iniciado o processo de treinamento e teste dos modelos

#### Support Vector Machine

In [ ]:
from sklearn.svm import SVC

clf = SVC(gamma='auto', random_state=42)
clf.fit(X_train, y_train)

In [ ]:
y_predictions['SVM'] = clf.predict(X_test)

In [ ]:
accuracy_score(y_test, y_predictions['SVM'])

#### Random Forest

In [ ]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

In [ ]:
y_predictions['Random Forest'] = rf.predict(X_test)

In [ ]:
accuracy_score(y_test, y_predictions['Random Forest'])

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_predictions['Random Forest'])

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=rf.classes_)
fig, ax = plt.subplots(figsize=(7.4, 6))

disp.plot(cmap='binary', text_kw={"fontsize": 14}, ax=ax)

plt.title("ReName Model - Random Forest", fontsize=16)
plt.xlabel("Predicted label", fontsize=16)
plt.ylabel("True label", fontsize=16)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

cbar = ax.images[-1].colorbar
cbar.ax.tick_params(labelsize=14)

plt.show()

In [ ]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

y_predictions['Random Forest'] = rf.predict(X_test)

In [ ]:
from sklearn.pipeline import Pipeline
from joblib import dump

In [ ]:
dump(vectorizer, 'vectorizer.joblib', compress=True)

In [ ]:
dump(rf, 'rf_corretor.joblib', compress=True)

##### 6.3) Explicabilidade random forest

In [ ]:
import lime
import numpy as np
import lime.lime_tabular

In [ ]:
predict_fn_rf = lambda x: rf.predict_proba(x).astype(float)
X = X_train.values
# explainer = lime.lime_tabular.LimeTabularExplainer(X,feature_names = X_train.columns, discretize_continuous=True)
explainer = lime.lime_tabular.LimeTabularExplainer(X,feature_names = X_train.columns)

In [ ]:
X_test

In [ ]:
choosen_instance = X_test.values[0]
exp = explainer.explain_instance(choosen_instance, predict_fn_rf,num_features=10)
exp.show_in_notebook(show_all=False)

In [ ]:
alfabeto = list(string.ascii_uppercase)
alfabeto_bigrams = [''.join(bigram) for bigram in list(product(alfabeto, repeat=2))]

In [ ]:
teste = X_test.iloc[[0]]

df_result = teste.merge(tn_vector_bigrams, on=alfabeto_bigrams, how='left')
df_result

In [ ]:
choosen_instance = X_test.values[40]
exp = explainer.explain_instance(choosen_instance, predict_fn_rf,num_features=10)
exp.show_in_notebook(show_all=False)
plt.savefig('2-explicabilidade_etapa_treino.png')

In [ ]:
teste = X_test.iloc[[40]]

df_result = teste.merge(fn_vector_bigrams, on=alfabeto_bigrams, how='left')
df_result

In [ ]:
for i in range(len(X_test)):  # Não precisa do range(0, ...)
    teste = X_test.iloc[[i]]

    df_result = teste.merge(fn_vector_bigrams, on=alfabeto_bigrams, how='left')

    if not df_result[df_result['palavra'] == "TORRES"].empty:
        print(i)

#### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

In [ ]:
y_predictions['Logistic Regression'] = model.predict(X_test)

In [ ]:
accuracy_score(y_test, y_predictions['Logistic Regression'])

#### Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)

In [ ]:
y_predictions['Gradient Boosting'] = gb.predict(X_test)

In [ ]:
accuracy_score(y_test, y_predictions['Gradient Boosting'])

## PASSO 7) Apresentação dos resultados

### 7.1) Confusion Matrix

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(8, 6))

for model_name, y_pred, ax in zip(y_predictions.keys(), y_predictions.values(), axs.flat):
    cm = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(y_test, y_pred), display_labels=clf.classes_, )
    cm.plot(ax=ax, cmap='binary', values_format='d')
    ax.grid(False)
    ax.set_title(model_name)
plt.tight_layout()
# plt.savefig('imagens/matriz_confusao.png', bbox_inches='tight', dpi = 300)

In [ ]:
dfs = {}
for model_name, y_pred in y_predictions.items():
    report = {}
    report['accuracy'] = accuracy_score(y_test, y_pred)
    report['precision'] = precision_score(y_test, y_pred)
    report['recall'] = recall_score(y_test, y_pred)
    report['f1-score'] = f1_score(y_test, y_pred)
    dfs[model_name] = pd.DataFrame(report, index=[model_name])

### 7.2) Metrics - ACCURACY, PRECISION, RECALL, F1-SCORE

In [ ]:
concatenated_df = pd.concat(dfs.values())
concatenated_df

In [ ]:
plt.style.use('bmh')

### 7.3) Comparação métricas dos algoritmos

In [ ]:
concatenated_df.T.plot.bar(figsize=(7, 3), ylim=(0, 1.0), color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'], grid=1).legend(bbox_to_anchor=(1, 1))
plt.xticks(rotation=0)
plt.tight_layout()

## PASSO 8) Realização da Validação cruzada

In [ ]:
def validacao_cruzada(X_train, y_train, models):
    cv_results= {}
    scores = {}

    for model_name, model in models.items():
        score = cross_val_score(estimator=model, X=X_train, y=y_train, cv=10)
        media = score.mean()
        desvio = score.std()
        scores[model_name] = score
        cv_results[model_name] = {'Validação Cruzada': "%.5f" % media, 'Desvio Padrão': "%.5f" % desvio}
        
    return cv_results, scores

In [ ]:
cv_results, scores = validacao_cruzada(X_train, y_train,{
    'SVC': SVC(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
})

#### comentários a cerca do boxplot


Observando o resultado da validação cruzada, podemos observar o comportamento do algoritmo quando o mesmo se depara com diferentes conjuntos de dados para o treinamento.

Destaque para o Random Forest que possuiu pouca variância se comparado com os outros modelos.

In [ ]:
plt.figure(figsize=(10, 4))
plt.boxplot(scores.values(), tick_labels=scores.keys(), patch_artist=True)
plt.grid(True)
plt.title('Validação Cruzada')
plt.ylabel('Acurácia')
# plt.savefig('images/boxplot.png', bbox_inches='tight', dpi = 300)
plt.show()

### 8.1) Média e Desvio Padrão dos resultados obtidos na validação cruzada

está em ordem desc

In [ ]:
scores

####  Media ( validaçao cruzada )

In [ ]:
np.mean(scores['Random Forest'])

In [ ]:
np.mean(scores['Logistic Regression'])

In [ ]:
np.mean(scores['Gradient Boosting'])

In [ ]:
np.mean(scores['SVC'])

#### Desvio Padrão ( validaçao cruzada )

In [ ]:
np.std(scores['Random Forest'])

In [ ]:
np.std(scores['Logistic Regression'])

In [ ]:
np.std(scores['Gradient Boosting'])

In [ ]:
np.std(scores['SVC'])